# Adjoints from scratch

The whole of reverse-mode automatic differentiation rests on one move.
This notebook builds it from nothing — no tensors, no library, no linear
algebra vocabulary. Three ideas, in order: a derivative is a *sensitivity*;
a computation is a *graph of paths*; and the backward pass is *one
regrouping of a sum*. The last one is the star.

In [1]:
import numpy as np


def wiggle(f, x, eps=1e-6):
    # numerical sensitivity: how much does f(x) move per unit wiggle of x?
    return (f(x + eps) - f(x - eps)) / (2 * eps)

## 1. A derivative is a sensitivity

Wiggle the input by a tiny ε; the output moves by (slope)·ε. The slope is
the exchange rate between input-wiggles and output-wiggles.

In [2]:
def f(x):
    return x * x


print("slope of x² at 3:", wiggle(f, 3.0), " — wiggle in, 6x wiggle out")

slope of x² at 3: 6.000000000838668  — wiggle in, 6x wiggle out


## 2. Chains multiply

Feed f into g and the exchange rates compose: a wiggle is converted twice.
That is the chain rule, and it is just multiplication.

In [3]:
def g(y):
    return 3 * y + 1


print("slope of g∘f at 3:", wiggle(lambda x: g(f(x)), 3.0))
print("product of slopes:", wiggle(g, f(3.0)) * wiggle(f, 3.0))

slope of g∘f at 3: 18.000000002516003
product of slopes: 17.999999999702936


## 3. Reuse adds — the path-sum picture

Compute y = x·x, thinking of the two x's as two *uses* of one value:

        x ──┬──▶ (·) ──▶ y
            └──────┘

A wiggle in x reaches y along **two paths** (once through each use), and
the effects **add**: slope = x + x = 2x. In general:

> sensitivity of output to input = Σ over every path (Π of slopes along it)

Every rule that follows is a way of organizing this sum.

In [4]:
print("slope of x·x at 5:", wiggle(lambda x: x * x, 5.0), " = 5 + 5, one per path")

slope of x·x at 5: 10.00000000139778  = 5 + 5, one per path


## 4. Why backwards?

The path-sum can be evaluated from either end. Sweeping *forward* answers
"how does everything depend on this one input?" — one sweep **per input**.
Sweeping *backward* answers "how does this one output depend on
everything?" — one sweep **per output**. Training a model has a million
inputs (the parameters) and one output (the loss): backward wins by a
factor of a million. That is the entire motivation for reverse mode.

## 5. Ops on lists of numbers, and the bookkeeping problem

Real programs move *lists* of numbers. Suppose the backward sweep has
reached some op: we already know how much the loss cares about each
**output** slot — a list of sensitivities ȳ. The op's job in the backward
sweep: convert ȳ into sensitivities x̄ for each **input** slot.

The bookkeeping device is the pairing

    ⟨a, b⟩ = Σᵢ aᵢ·bᵢ        "total effect of wiggles b under sensitivities a"

A wiggle dx in the inputs produces output wiggles Op(dx), so the loss moves
by ⟨ȳ, Op(dx)⟩. We want x̄ such that the SAME total effect reads
⟨x̄, dx⟩ — sensitivities expressed at the input.

In [5]:
def ip(a, b):
    return float(np.sum(np.asarray(a) * np.asarray(b)))

## 6. The one rule: swap the order of summation

Write the total effect as a double sum and regroup it around the inputs.
If output j moves by Σᵢ Sⱼᵢ·dxᵢ (each output a weighted mix of input
wiggles), then

    total effect = Σⱼ ȳⱼ · (Σᵢ Sⱼᵢ dxᵢ)          — grouped by outputs
                 = Σᵢ ( Σⱼ Sⱼᵢ ȳⱼ ) · dxᵢ        — grouped by inputs

so    **x̄ᵢ = Σⱼ Sⱼᵢ ȳⱼ**.

That is everything. The backward op — call it Op† — is *defined* by

    ⟨ȳ, Op(dx)⟩ = ⟨Op†(ȳ), dx⟩

"move the op to the other side of the pairing." Deriving any Op† is the
same three steps: write the left side as a sum over outputs, substitute
what the op does, regroup around the inputs.

And there is a beautiful way to *test* an adjoint that needs no loss
function and no finite differences: the defining equation itself, checked
on random vectors.

In [6]:
def pairing_check(op, adj, nin, nout, label):
    rng = np.random.default_rng(0)
    for _ in range(5):
        dx, ybar = rng.standard_normal(nin), rng.standard_normal(nout)
        assert np.isclose(ip(ybar, op(dx)), ip(adj(ybar), dx))
    print(f"{label:<28} ⟨ȳ, Op dx⟩ == ⟨Op†ȳ, dx⟩  ok")

## 7. The whole zoo, by hand

Each example: what the op does, the regrouped sum, the adjoint it forces.

**sum** — y = x₁+x₂+x₃ (one output). Effect: ȳ·(dx₁+dx₂+dx₃) =
Σᵢ ȳ·dxᵢ, so x̄ = (ȳ, ȳ, ȳ): **the adjoint of summing is broadcasting**.

In [7]:
pairing_check(lambda x: np.array([x.sum()]), lambda y: np.repeat(y, 3), 3, 1, "sum ⊣ broadcast")

sum ⊣ broadcast              ⟨ȳ, Op dx⟩ == ⟨Op†ȳ, dx⟩  ok


**broadcast** — yⱼ = x for j = 1..3. Effect: Σⱼ ȳⱼ·dx, so x̄ = Σⱼ ȳⱼ:
**the adjoint of broadcasting is summing**. The two are a pair — each is
the other's backward op.

In [8]:
pairing_check(lambda x: np.repeat(x, 3), lambda y: np.array([y.sum()]), 1, 3, "broadcast ⊣ sum")

broadcast ⊣ sum              ⟨ȳ, Op dx⟩ == ⟨Op†ȳ, dx⟩  ok


**selection** — y = (x₂, x₃). Slots that were never read get zero
sensitivity: **the adjoint of slicing is zero-padding**.

In [9]:
pairing_check(lambda x: x[1:3], lambda y: np.concatenate([[0.0], y, [0.0]]), 4, 2, "slice ⊣ zero-pad")

slice ⊣ zero-pad             ⟨ȳ, Op dx⟩ == ⟨Op†ȳ, dx⟩  ok


**rearrangement** — y = reverse(x). Sensitivities ride back along the same
relabeling: **adjoint of a rearrangement is its inverse**.

In [10]:
pairing_check(lambda x: x[::-1], lambda y: y[::-1], 4, 4, "reverse ⊣ reverse")

reverse ⊣ reverse            ⟨ȳ, Op dx⟩ == ⟨Op†ȳ, dx⟩  ok


**overlap** — yⱼ = xⱼ + xⱼ₊₁ (a two-tap sliding window). Regroup:

    ȳ₁(dx₁+dx₂) + ȳ₂(dx₂+dx₃) = ȳ₁dx₁ + (ȳ₁+ȳ₂)dx₂ + ȳ₂dx₃

Middle slots were read twice, so they collect twice: **the adjoint of an
overlapping read is overlap-add** — convolution's backward pass in
miniature.

In [11]:
def overlap(x):
    return x[:-1] + x[1:]


def overlap_add(y):
    x = np.zeros(len(y) + 1)
    x[:-1] += y
    x[1:] += y
    return x


pairing_check(overlap, overlap_add, 4, 3, "overlap ⊣ overlap-add")

overlap ⊣ overlap-add        ⟨ȳ, Op dx⟩ == ⟨Op†ȳ, dx⟩  ok


**pointwise nonlinearity** — yᵢ = xᵢ². Not linear, but its *wiggle
behavior at a point* is: dyᵢ = 2xᵢ·dxᵢ. Nonlinear ops enter the machinery
through their local slopes; the adjoint multiplies by the same slopes.

In [12]:
x0 = np.array([1.0, -2.0, 3.0])
pairing_check(lambda dx: 2 * x0 * dx, lambda y: 2 * x0 * y, 3, 3, "square (linearized)")

square (linearized)          ⟨ȳ, Op dx⟩ == ⟨Op†ȳ, dx⟩  ok


## 8. The atom underneath: copy ⊣ add

Look again at every example — each is index bookkeeping over one atom:

- **using a value twice (copy)**: the two downstream sensitivities **add**;
- **adding two values**: the sensitivity is **copied** to both.

Fan-out ⊣ fan-in. Section 3's two paths, sum's broadcast, overlap's
double-collection — all the same atom wearing different indices.

## 9. Reverse mode is the swap, performed mechanically

Walk the program backwards. Start from the loss with sensitivity 1. At
each op, push the running sensitivities through Op†. Where a value fed
several ops, add what comes back. A full backward pass, by hand:

In [13]:
rng = np.random.default_rng(1)
xv, wv = rng.standard_normal(4), rng.standard_normal(4)

# forward:  a = x*w ;  b = a² ;  L = sum(b)
a = xv * wv
L = float((a**2).sum())

# backward, one adjoint at a time:
bbar = np.ones(4)  # sum† : broadcast the seed 1
abar = 2 * a * bbar  # square† : multiply by the local slope
xbar = wv * abar  # mul† : route through the other operand


def loss_with(i, t):
    x2 = xv.copy()
    x2[i] = t
    return float(((x2 * wv) ** 2).sum())


fd = np.array([wiggle(lambda t: loss_with(i, t), xv[i]) for i in range(4)])
print("hand backward:", np.round(xbar, 6))
print("finite diff:  ", np.round(fd, 6))

hand backward: [ 0.566529  0.327415  0.190542 -0.880148]
finite diff:   [ 0.566529  0.327415  0.190542 -0.880148]


## 10. Bridge: this is exactly what the library does

tensorlib's layout ops are the zoo at scale — `repeat` is broadcast,
`slice`/`pad` the selection pair, `window`/`stencil` the overlap,
relabelings ride back unchanged. Notebook 05's cheat sheet is this
notebook's rule applied to every instruction, and `grad` performs the
sum-swap mechanically over whole programs:

In [14]:
from nbhelp import show  # puts tensorlib on sys.path
from tensorlib import Tensor
from tensorlib.autodiff import grad
from tensorlib.ir import Instr, Program
from tensorlib.ir import run as run_ir

p = Program(
    (
        Instr("x", "input", (), {}),
        Instr("w", "input", (), {}),
        Instr("a", "pointwise", ("x", "w"), {"f": "mul"}),
        Instr("b", "pointwise", ("a", "a"), {"f": "mul"}),
        Instr("L", "reduce", ("b",), {"f": "sum", "dims": ("i",)}),
    )
)
X, W = Tensor.from_numpy(xv, ("i",)), Tensor.from_numpy(wv, ("i",))
joint, grads = grad(p, "L", {"x": X, "w": W})
env = run_ir(joint, {"x": X, "w": W})
print("tensorlib grad:", np.round(env[grads["x"]].to_numpy(), 6))
print("hand backward: ", np.round(xbar, 6))
show(env[grads["x"]], "the gradient is an ordinary tensor")

tensorlib grad: [ 0.566529  0.327415  0.190542 -0.880148]
hand backward:  [ 0.566529  0.327415  0.190542 -0.880148]
-- the gradient is an ordinary tensor
Tensor[float64] on Buffer(32B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i            8      0      4      4  
  numel=4  footprint=(0, 32)  injectivity=injective


---
The summary fits in one sentence: **sensitivities flow backwards through
each operation's adjoint — defined by ⟨ȳ, Op dx⟩ = ⟨Op†ȳ, dx⟩, i.e. one
swap of a double sum — and add wherever paths merge.** Everything else is
indices.